# T01 — Integridade e Cobertura das Bases FRTB

**Tipo de teste:** qualidade de dados, integridade referencial e reconciliação operacional.  
**População:** posições, fluxos e fluxos por cenário da mesma data-base.  
**Responsável:** Validação Independente de Modelos.  

## 1. Objetivo, hipóteses e limitações

Verificar: (i) disponibilidade e estrutura dos arquivos; (ii) preenchimento dos identificadores; (iii) coerência das datas-base; (iv) duplicidade exata de linhas; (v) correspondência entre conjuntos de identificadores; e (vi) perfil das posições sem fluxos.

**Hipóteses:** `IDPos` identifica uma posição de forma consistente entre as três bases; os arquivos representam uma única data-base comparável; múltiplas linhas por posição são permitidas (fluxos, pernas, categorias contábeis ou cenários).

**Limitações:** a correspondência de `IDPos` não comprova completude econômica, elegibilidade FRTB, cálculo correto de sensibilidades ou capital. Ausência de fluxo convencional não equivale a exclusão regulatória. O teste de duplicidade exata não substitui avaliação de unicidade por chave de negócio.

**Critério de avaliação:** desvios são registrados e classificados para investigação. A conclusão final depende de evidências documentais das exceções, não apenas da ausência de erros de execução.

## 2. Configuração e carregamento

Estrutura esperada:

```text
FRTB/
├── dados/
│   ├── base posicoes.xlsx
│   ├── fluxos.xlsx
│   └── fluxos por cenario.xlsx
└── Valid_FRTB/
    └── T01_CobAnd.ipynb
```

Se os arquivos tiverem nomes diferentes, altere apenas `ARQUIVOS`. O código não publica os dados nem imprime identificadores individuais.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Notebook em FRTB/Valid_FRTB; funciona também quando iniciado na pasta FRTB.
PASTA_ATUAL = Path.cwd().resolve()
PASTA_RAIZ = PASTA_ATUAL.parent if PASTA_ATUAL.name.lower() == "valid_frtb" else PASTA_ATUAL
PASTA_DADOS = PASTA_RAIZ / "dados"
PASTA_RESULTADOS = PASTA_ATUAL / "resultados"
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

ARQUIVOS = {
    "posicoes": "base posicoes.xlsx",
    "fluxos": "fluxos.xlsx",
    "cenarios": "fluxos por cenario.xlsx",
}
DATA_BASE_ESPERADA = None  # Exemplo: "2026-09-30"; None = sem data pré-fixada

print("Dados:", PASTA_DADOS)
print("Resultados:", PASTA_RESULTADOS)
for nome, arquivo in ARQUIVOS.items():
    caminho = PASTA_DADOS / arquivo
    print(f"{nome}: {'ENCONTRADO' if caminho.is_file() else 'AUSENTE'} — {caminho.name}")

In [ ]:
def carregar_excel(caminho):
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")
    df = pd.read_excel(caminho, dtype={"IDPos": "string"})
    df.columns = df.columns.astype(str).str.strip()
    if df.columns.duplicated().any():
        raise ValueError(f"Colunas duplicadas após normalização em {caminho.name}")
    if "IDPos" not in df.columns:
        raise ValueError(f"IDPos ausente em {caminho.name}")
    df["IDPos"] = df["IDPos"].astype("string").str.strip()
    df.loc[df["IDPos"].eq(""), "IDPos"] = pd.NA
    return df

bases = {nome: carregar_excel(PASTA_DADOS / arquivo) for nome, arquivo in ARQUIVOS.items()}
pos, flx, cen = bases["posicoes"], bases["fluxos"], bases["cenarios"]
print("Arquivos carregados com sucesso.")

## 3. Diagnóstico estrutural

`OK` significa apenas que os campos mínimos configurados estão presentes e que `IDPos` não está vazio. Não significa aprovação regulatória ou econômica. Campos adicionais devem ser avaliados nos testes posteriores.

In [ ]:
CAMPOS_MINIMOS = {
    "posicoes": ["IDPos", "DtPos", "CodProduto", "CodCarteiraBN", "CodCategCont"],
    "fluxos": ["IDPos", "IdFluxo", "flx_VlrPresente"],
    "cenarios": ["IDPos", "DtPos", "CodComponent", "CodClasse"],
}

def ids_validos(df):
    return set(df["IDPos"].dropna().astype(str))

diagnostico = []
for nome, df in bases.items():
    faltantes = sorted(set(CAMPOS_MINIMOS[nome]) - set(df.columns))
    nulos = int(df["IDPos"].isna().sum())
    diagnostico.append({
        "Base": nome,
        "Linhas": len(df),
        "Colunas": len(df.columns),
        "IDs_unicos": len(ids_validos(df)),
        "IDs_nulos": nulos,
        "Campos_ausentes": ", ".join(faltantes),
        "Status": "OK" if not faltantes and nulos == 0 else "INVESTIGAR",
    })
df_diagnostico = pd.DataFrame(diagnostico)
display(df_diagnostico)

## 4. Cobertura e reconciliação de identificadores

Definições:

\[
\operatorname{Cobertura}_{P\to F}=100\frac{|P\cap F|}{|P|},\qquad
\operatorname{Cobertura}_{P\to C}=100\frac{|P\cap C|}{|P|}.
\]

Onde `P`, `F` e `C` são conjuntos de `IDPos` das bases de posições, fluxos e cenários. Esta é uma **métrica de correspondência de identificadores**, não uma medida de cobertura econômica de risco.

In [ ]:
ids_pos = ids_validos(pos)
ids_flx = ids_validos(flx)
ids_cen = ids_validos(cen)

def percentual(parte, total):
    return 100 * parte / total if total else np.nan

cobertura = pd.DataFrame([
    {"Indicador": "Posições com fluxos", "Quantidade": len(ids_pos & ids_flx),
     "Total_referencia": len(ids_pos), "Cobertura_pct": percentual(len(ids_pos & ids_flx), len(ids_pos))},
    {"Indicador": "Posições com cenários", "Quantidade": len(ids_pos & ids_cen),
     "Total_referencia": len(ids_pos), "Cobertura_pct": percentual(len(ids_pos & ids_cen), len(ids_pos))},
])

excecoes_ids = {
    "Pos_Sem_Fluxos": ids_pos - ids_flx,
    "Pos_Sem_Cenarios": ids_pos - ids_cen,
    "Fluxos_Sem_Pos": ids_flx - ids_pos,
    "Cenarios_Sem_Pos": ids_cen - ids_pos,
}
resumo_excecoes = pd.DataFrame([
    {"Excecao": nome, "IDs_unicos": len(conjunto)}
    for nome, conjunto in excecoes_ids.items()
])
display(cobertura.round(4))
display(resumo_excecoes)

## 5. Duplicidades e integridade temporal

A repetição de `IDPos` é esperada em bases com múltiplos fluxos ou cenários. Por isso, distinguimos **duplicidade exata de registro** de **repetição legítima de identificador**.

Para as datas, comparamos datas normalizadas (`YYYY-MM-DD`), sem impor que a data-base do arquivo seja a data atual.

In [ ]:
duplicidades = []
for nome, df in bases.items():
    # Comparação por todas as colunas: útil como alerta, mas não define chave de negócio.
    duplicidades.append({
        "Base": nome,
        "Linhas_duplicadas_exatas": int(df.duplicated(keep=False).sum()),
        "Linhas_com_ID_repetido": int(df["IDPos"].duplicated(keep=False).sum()),
        "IDs_com_repeticao": int(df.loc[df["IDPos"].duplicated(keep=False), "IDPos"].nunique()),
    })
df_duplicidades = pd.DataFrame(duplicidades)
display(df_duplicidades)

datas = []
conjuntos_datas = {}
for nome, df in bases.items():
    if "DtPos" not in df.columns:
        datas.append({"Base": nome, "Registros": len(df), "Datas_invalidas": np.nan,
                      "Datas_distintas": np.nan, "Datas": "CAMPO NÃO DISPONÍVEL"})
        continue
    serie = pd.to_datetime(df["DtPos"], errors="coerce", dayfirst=True).dt.normalize()
    valores = sorted(set(serie.dropna().dt.strftime("%Y-%m-%d")))
    conjuntos_datas[nome] = set(valores)
    datas.append({"Base": nome, "Registros": len(df),
                  "Datas_invalidas": int(serie.isna().sum()),
                  "Datas_distintas": len(valores), "Datas": ", ".join(valores[:10])})
df_datas = pd.DataFrame(datas)
display(df_datas)

if conjuntos_datas:
    datas_coincidentes = len({frozenset(v) for v in conjuntos_datas.values()}) == 1
    print("Datas-base coincidentes entre bases com DtPos:", datas_coincidentes)
if DATA_BASE_ESPERADA is not None:
    esperada = pd.Timestamp(DATA_BASE_ESPERADA).strftime("%Y-%m-%d")
    print("Todas as bases com DtPos contêm exclusivamente a data esperada:",
          all(v == {esperada} for v in conjuntos_datas.values()))

## 6. Caracterização das exceções

Caracterizar posições sem fluxos por produto, carteira e categoria contábil. A explicação recebida da área de Riscos de Mercado deve ser formalizada em documento ou registro rastreável.

**Importante:** o código não exclui posições da população de referência. A ausência de fluxo convencional não prova ausência de exposição a risco de mercado nem inelegibilidade FRTB.

In [ ]:
pos_sem_fluxos = pos.loc[pos["IDPos"].isin(excecoes_ids["Pos_Sem_Fluxos"])].copy()
print("IDs únicos sem fluxos:", pos_sem_fluxos["IDPos"].nunique())
print("Linhas de posições associadas:", len(pos_sem_fluxos))
print("Linhas por ID (distribuição):")
display(pos_sem_fluxos.groupby("IDPos").size().value_counts().sort_index().rename_axis("Linhas_por_ID").reset_index(name="Quantidade_IDs"))

for campo in ["CodProduto", "CodCarteiraBN", "CodCategCont"]:
    if campo in pos_sem_fluxos.columns:
        print(f"Distribuição de {campo}:")
        display(pos_sem_fluxos[campo].fillna("<NULO>").value_counts(dropna=False).rename_axis(campo).reset_index(name="Linhas"))

# Resumo anônimo das exceções para documentação pública; não imprimir IDs reais.


## 7. Consolidação e critérios de conclusão

A classificação automática abaixo indica **pendências de investigação**, não um parecer final automático. O resultado `OK` no diagnóstico estrutural não significa que todas as posições foram economicamente capturadas.

In [ ]:
alertas = []
if (df_diagnostico["Status"] != "OK").any():
    alertas.append("Campos mínimos ausentes ou IDPos nulo")
if (df_duplicidades["Linhas_duplicadas_exatas"] > 0).any():
    alertas.append("Existem linhas exatamente duplicadas")
if (df_datas["Datas_invalidas"].fillna(0) > 0).any():
    alertas.append("Datas-base inválidas")
if conjuntos_datas and not datas_coincidentes:
    alertas.append("Datas-base não coincidentes")
if any(excecoes_ids.values()):
    alertas.append("Exceções de correspondência entre bases")

print("RESULTADO AUTOMÁTICO T01:")
print("SEM ALERTAS AUTOMÁTICOS" if not alertas else "INVESTIGAÇÃO NECESSÁRIA")
for a in alertas:
    print("-", a)
print("Observação: conclusão final requer julgamento do validador e evidências funcionais.")

In [ ]:
# Exportar evidências agregadas sem identificadores individuais.
saida = PASTA_RESULTADOS / "T01_integridade_cobertura_resultados.xlsx"
with pd.ExcelWriter(saida, engine="openpyxl") as writer:
    df_diagnostico.to_excel(writer, sheet_name="Diagnostico", index=False)
    cobertura.to_excel(writer, sheet_name="Cobertura", index=False)
    resumo_excecoes.to_excel(writer, sheet_name="Excecoes_Resumo", index=False)
    df_duplicidades.to_excel(writer, sheet_name="Duplicidades", index=False)
    df_datas.to_excel(writer, sheet_name="Datas", index=False)
    for campo, aba in [("CodProduto", "Perfil_Produto"),
                       ("CodCarteiraBN", "Perfil_Carteira"),
                       ("CodCategCont", "Perfil_Contabil")]:
        if campo in pos_sem_fluxos.columns:
            perfil = pos_sem_fluxos[campo].fillna("<NULO>").value_counts().rename_axis(campo).reset_index(name="Linhas")
            perfil.to_excel(writer, sheet_name=aba, index=False)
print("Relatório agregado gerado:", saida)
print("ATENÇÃO: dados e perfis de produtos/carteiras podem ser confidenciais. Não publique esta planilha no GitHub.")

## 8. Parecer técnico do validador

**Resultado observado na execução analisada:** foram identificados 17.816 IDs únicos em posições, 17.625 em fluxos e 17.816 em cenários. A cobertura de posições com fluxos foi de aproximadamente 98,93%, enquanto a correspondência com cenários alcançou 100%. Foram observados 191 IDs de posições sem fluxos, correspondentes a 382 linhas, concentrados em instrumentos de Hedge Accounting DDI.

**Interpretação e evidência funcional:** segundo esclarecimento da área de Riscos de Mercado, esses instrumentos não geram fluxos convencionais na estrutura analisada. Essa informação fornece justificativa funcional para a diferença de correspondência, **sujeita à formalização e à verificação de sua aplicação ao universo de posições analisado**.

**Conclusão:** os controles de integridade estrutural e correspondência de identificadores não evidenciaram, na execução analisada, falhas estruturais relevantes. O resultado do T01 é **satisfatório com ressalva documental**, condicionado à formalização da regra de tratamento dos instrumentos sem fluxos. O teste não permite concluir que o capital FRTB esteja correto, que todas as exposições econômicas estejam cobertas ou que os instrumentos sem fluxos possam ser excluídos do cálculo regulatório.

**Encaminhamento:** avançar ao T02 (classificação de posições e fatores de risco), mantendo rastreabilidade das exceções, documentação da regra funcional e avaliação posterior da elegibilidade regulatória.

> **Controle de versão:** este parecer descreve a execução cujos números foram analisados. Se os dados mudarem, revisar o texto com base nas tabelas recalculadas antes de emitir novo parecer.

### Questões de aprofundamento

1. Qual chave composta identifica univocamente um fluxo (`IDPos`, `IdFluxo`, categoria contábil, perna etc.)?
2. Como comprovar que a população extraída corresponde ao universo completo de operações elegíveis ao FRTB?
3. Qual evidência independente confirma o tratamento dos instrumentos de Hedge Accounting DDI?
4. É possível reconciliar o valor presente dos fluxos com a posição por categoria contábil e carteira? (Tema para um teste financeiro específico.)